# Training DDPM on CIFAR-10

This notebook trains $\varepsilon_\theta(x_t, t)$ with the simplified objective from Ho et al.,
[Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239) (2020), Algorithm 1:

$$L_\text{simple} = \|\varepsilon - \varepsilon_\theta(x_t, t)\|_2^2$$

It follows [`05_check_unet.ipynb`](05_check_unet.ipynb). The schedule and $L_\text{simple}$ come from
[`02_diffusion.py`](02_diffusion.py); the U-Net comes from [`04_unet.py`](04_unet.py).
**Do not start the long GPU run until the one-batch overfit loss drops.** Reverse-process
sampling stays in [`07_sample_eval.ipynb`](07_sample_eval.ipynb).

**Train/test convention.** This is unconditional generation. We train on the official 50k training
images and ignore labels. The 10k test set is a later qualitative / FID reference, not a classification
hold-out.


## 1. Setup


In [ ]:
import copy
import math
import importlib
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Numbered .py files are not valid import identifiers.
sys.modules["diffusion"] = importlib.import_module("02_diffusion")
sys.modules["model"] = importlib.import_module("04_unet")
from diffusion import DiffusionSchedule, build_schedule, simple_loss
from model import UNet


@dataclass(frozen=True)
class Config:
    seed: int = 42
    data_dir: Path = Path("dataset")
    checkpoint_dir: Path = Path("checkpoints")
    image_size: int = 32
    in_channels: int = 3
    num_steps: int = 1000
    beta_start: float = 1e-4
    beta_end: float = 0.02
    # Paper-width CIFAR-10 U-Net (Tier B).
    base_channels: int = 128
    channel_mults: tuple[int, ...] = (1, 2, 2, 2)
    num_res_blocks: int = 2
    attention_resolutions: tuple[int, ...] = (16,)
    dropout: float = 0.1
    # Physical 32, four accumulation steps -> effective 128, as in the paper.
    batch_size: int = 32
    grad_accum_steps: int = 4
    learning_rate: float = 2e-4
    ema_decay: float = 0.9999
    max_steps: int = 100_000
    log_every: int = 50
    checkpoint_every: int = 5_000


cfg = Config()

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.benchmark = True

# AMP is a CUDA path. On CPU the scaler is created but disabled so checkpoint keys stay the same.
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
cfg.checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Timed training run. Set TRAIN_MINUTES to 0 to use max_steps only.
RUN_PRODUCTION_TRAINING = True
TRAIN_MINUTES = 60
RESUME_CHECKPOINT = cfg.checkpoint_dir / "ddpm_cifar10_latest.pt"

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"AMP enabled: {use_amp}")
print(f"Production training: {RUN_PRODUCTION_TRAINING}, time cap: {TRAIN_MINUTES} min")
print(cfg)


## 2. Training data

Same transform as [`01_dataset.ipynb`](01_dataset.ipynb): random horizontal flip, then map
$[0, 1]$ to $[-1, 1]$. Labels are loaded and discarded.


In [ ]:
diffusion_transform = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)

data_ready = (cfg.data_dir / "cifar-10-batches-py").exists()
train_dataset = datasets.CIFAR10(
    root=cfg.data_dir,
    train=True,
    download=not data_ready,
    transform=diffusion_transform,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=device.type == "cuda",
    drop_last=True,
)

images, labels = next(iter(train_loader))
print(f"Train images: {len(train_dataset):,}")
print(f"Batch shape: {tuple(images.shape)}")
print(f"Value range: [{images.min():.3f}, {images.max():.3f}]")
print(f"Ignoring {labels.numel()} class labels (unconditional DDPM).")

assert images.shape == (cfg.batch_size, cfg.in_channels, cfg.image_size, cfg.image_size)
assert images.min() >= -1.01 and images.max() <= 1.01


## 3. Schedule and U-Net

Imported from [`02_diffusion.py`](02_diffusion.py) and [`04_unet.py`](04_unet.py).


In [ ]:
schedule = build_schedule(cfg.num_steps, cfg.beta_start, cfg.beta_end).to(device)
model = UNet(cfg).to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parameters: {parameter_count:,} ({parameter_count / 1e6:.2f} M)")
print(f"alpha_bar[0]={schedule.alpha_bars[0].item():.6f}, alpha_bar[-1]={schedule.alpha_bars[-1].item():.6e}")

assert 30e6 < parameter_count < 42e6
assert schedule.posterior_variance[0].item() == 0.0


## 4. $L_\text{simple}$

Sample $t$ uniformly, draw $\varepsilon \sim \mathcal{N}(0, I)$, form $x_t$ with `q_sample`, and
regress the U-Net onto that same $\varepsilon$. The loss lives in [`02_diffusion.py`](02_diffusion.py).
Zero-init on the output convolution makes the first loss close to $\mathrm{MSE}(0, \varepsilon) \approx 1$.


In [ ]:
probe_images = images[:2].to(device)
probe_loss = simple_loss(model, probe_images, schedule)
print(f"First L_simple (should be near 1): {probe_loss.item():.4f}")
assert probe_loss.ndim == 0
assert torch.isfinite(probe_loss)
assert 0.3 < probe_loss.item() < 3.0


## 5. One-batch overfit

This is the stop criterion from the report. We freeze one small batch and a fixed $(t, \varepsilon)$
so the graph has a single target. If this loss does not fall, the long run will not either.

A few extra steps with freshly sampled $t$ then confirm the stochastic training objective still runs.


In [ ]:
overfit_batch_size = 8 if device.type == "cuda" else 4
overfit_steps = 200 if device.type == "cuda" else 60

# Reuse the already-drawn training batch so the flip is applied once, then frozen.
overfit_images = images[:overfit_batch_size].to(device)
# Mixed timesteps so the time embedding is actually used, but the target stays fixed.
overfit_timesteps = torch.linspace(100, 800, overfit_batch_size, device=device).long()
overfit_noise = torch.randn_like(overfit_images)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
ema_model = copy.deepcopy(model)
ema_model.requires_grad_(False)
ema_model.eval()


@torch.no_grad()
def update_ema(ema_model: nn.Module, model: nn.Module, decay: float) -> None:
    for ema_param, param in zip(ema_model.parameters(), model.parameters()):
        ema_param.data.mul_(decay).add_(param.data, alpha=1.0 - decay)
    for ema_buffer, buffer in zip(ema_model.buffers(), model.buffers()):
        ema_buffer.copy_(buffer)


model.train()
overfit_losses = []
for step in range(1, overfit_steps + 1):
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", enabled=use_amp):
        loss = simple_loss(model, overfit_images, schedule, overfit_timesteps, overfit_noise)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    update_ema(ema_model, model, cfg.ema_decay)
    overfit_losses.append(float(loss.detach().cpu()))
    if step == 1 or step == overfit_steps or step % 10 == 0:
        print(f"overfit step {step:3d}/{overfit_steps}: L_simple={overfit_losses[-1]:.4f}")

start = sum(overfit_losses[:5]) / 5
end = sum(overfit_losses[-5:]) / 5
print(f"Fixed-target overfit: {start:.4f} -> {end:.4f}")
assert all(math.isfinite(value) for value in overfit_losses)
assert end < 0.5 * start, f"overfit did not drop enough: {start:.4f} -> {end:.4f}"

# Stochastic L_simple on the same images (random t, random noise), as in Algorithm 1.
model.train()
random_losses = []
for _ in range(8):
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", enabled=use_amp):
        loss = simple_loss(model, overfit_images, schedule)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    random_losses.append(float(loss.detach().cpu()))
print("Random-t losses on the same batch:", ", ".join(f"{value:.3f}" for value in random_losses))
assert all(math.isfinite(value) for value in random_losses)

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(range(1, len(overfit_losses) + 1), overfit_losses, color="#176b4d")
ax.set_xlabel("update")
ax.set_ylabel(r"$L_{\mathrm{simple}}$")
ax.set_title("One-batch overfit (fixed x0, t, noise)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 6. Checkpoints and resume

The checkpoint stores everything needed to continue: weights, EMA copy, optimizer, AMP scaler,
step counter, config, and loss history. After a save/load round-trip, a fresh model must match the
saved weights and take one more finite step.


In [ ]:
def config_payload(config: Config) -> dict:
    payload = asdict(config)
    payload["data_dir"] = str(config.data_dir)
    payload["checkpoint_dir"] = str(config.checkpoint_dir)
    return payload


def save_checkpoint(
    path: Path,
    model: nn.Module,
    ema_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    global_step: int,
    loss_history: list[float],
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model": model.state_dict(),
            "ema_model": ema_model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
            "global_step": global_step,
            "config": config_payload(cfg),
            "loss_history": loss_history,
        },
        path,
    )


def load_checkpoint(
    path: Path,
    model: nn.Module,
    ema_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
) -> tuple[int, list[float]]:
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    ema_model.load_state_dict(checkpoint["ema_model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scaler.load_state_dict(checkpoint["scaler"])
    return int(checkpoint["global_step"]), list(checkpoint["loss_history"])


overfit_path = cfg.checkpoint_dir / "overfit_smoke.pt"
save_checkpoint(
    overfit_path,
    model,
    ema_model,
    optimizer,
    scaler,
    global_step=overfit_steps,
    loss_history=overfit_losses,
)

# Fresh objects, then restore. If resume is wrong, parameter equality fails.
resumed_model = UNet(cfg).to(device)
resumed_ema = copy.deepcopy(resumed_model)
resumed_ema.requires_grad_(False)
resumed_optimizer = torch.optim.Adam(resumed_model.parameters(), lr=cfg.learning_rate)
resumed_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
loaded_step, loaded_history = load_checkpoint(
    overfit_path, resumed_model, resumed_ema, resumed_optimizer, resumed_scaler
)

ref = next(model.parameters()).detach()
got = next(resumed_model.parameters()).detach()
ema_ref = next(ema_model.parameters()).detach()
ema_got = next(resumed_ema.parameters()).detach()
print(f"Loaded step {loaded_step}, {len(loaded_history)} loss values, file={overfit_path}")
assert loaded_step == overfit_steps
assert loaded_history == overfit_losses
assert torch.allclose(ref, got)
assert torch.allclose(ema_ref, ema_got)

resumed_model.train()
resumed_optimizer.zero_grad(set_to_none=True)
with torch.autocast(device_type="cuda", enabled=use_amp):
    resume_loss = simple_loss(resumed_model, overfit_images, schedule, overfit_timesteps, overfit_noise)
resumed_scaler.scale(resume_loss).backward()
resumed_scaler.step(resumed_optimizer)
resumed_scaler.update()
print(f"One step after resume: {resume_loss.item():.4f}")
assert torch.isfinite(resume_loss)

# Keep training on the restored objects for the (optional) long run.
model, ema_model, optimizer, scaler = resumed_model, resumed_ema, resumed_optimizer, resumed_scaler


## 7. Production loop

Paper-faithful defaults: Adam $2 \times 10^{-4}$, dropout $0.1$, EMA $0.9999$, effective batch 128
via gradient accumulation. Set `RUN_PRODUCTION_TRAINING = True` after the overfit plot looks right.
`TRAIN_MINUTES` stops the loop on a wall-clock cap and writes a checkpoint; `0` means run to
`max_steps` only.

Sampling grids belong in [`07_sample_eval.ipynb`](07_sample_eval.ipynb). This loop only trains, logs,
and writes checkpoints.


In [ ]:
def infinite_loader(loader: DataLoader):
    while True:
        yield from loader


def train(
    model: nn.Module,
    ema_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    schedule: DiffusionSchedule,
    loader: DataLoader,
    start_step: int,
    max_steps: int,
    loss_history: list[float],
    max_seconds: float | None = None,
) -> list[float]:
    model.train()
    batches = infinite_loader(loader)
    optimizer.zero_grad(set_to_none=True)
    running = 0.0
    micro = 0
    step = start_step
    tick = time.perf_counter()
    started = time.perf_counter()

    def write_checkpoint(reason: str) -> None:
        save_checkpoint(
            RESUME_CHECKPOINT,
            model,
            ema_model,
            optimizer,
            scaler,
            global_step=step,
            loss_history=loss_history,
        )
        print(f"wrote {RESUME_CHECKPOINT} at step {step} ({reason})")

    while step < max_steps:
        batch_images, _ = next(batches)
        batch_images = batch_images.to(device, non_blocking=device.type == "cuda")
        with torch.autocast(device_type="cuda", enabled=use_amp):
            loss = simple_loss(model, batch_images, schedule) / cfg.grad_accum_steps
        scaler.scale(loss).backward()
        running += float(loss.detach().cpu())
        micro += 1
        if micro % cfg.grad_accum_steps != 0:
            continue

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        update_ema(ema_model, model, cfg.ema_decay)
        step += 1
        recorded = running
        loss_history.append(recorded)
        running = 0.0

        if step % cfg.log_every == 0 or step == start_step + 1:
            elapsed = time.perf_counter() - tick
            print(f"step {step}/{max_steps}  L_simple={recorded:.4f}  ({elapsed:.1f}s since last log)")
            tick = time.perf_counter()
        if step % cfg.checkpoint_every == 0 or step == max_steps:
            write_checkpoint("periodic")
        if max_seconds is not None and (time.perf_counter() - started) >= max_seconds:
            write_checkpoint(f"time cap {max_seconds / 60:.0f} min")
            print(f"Stopped by TRAIN_MINUTES at step {step}")
            break
    return loss_history


# Short throughput probe so a later GPU run can estimate wall-clock time from this machine.
model.train()
if device.type == "cuda":
    torch.cuda.synchronize()
bench_images = overfit_images
n_bench = 3
bench_start = time.perf_counter()
for _ in range(n_bench):
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", enabled=use_amp):
        bench_loss = simple_loss(model, bench_images, schedule)
    scaler.scale(bench_loss).backward()
    scaler.step(optimizer)
    scaler.update()
if device.type == "cuda":
    torch.cuda.synchronize()
bench_seconds = time.perf_counter() - bench_start
steps_per_second = n_bench / bench_seconds
hours_100k = cfg.max_steps / steps_per_second / 3600
print(f"Benchmark: {steps_per_second:.2f} optimizer steps/s on {device}")
print(f"Naive 100k-step estimate: {hours_100k:.1f} h (warm this up again on the GPU)")
if device.type == "cuda":
    print(f"Peak allocated memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")

if RUN_PRODUCTION_TRAINING:
    start_step = 0
    history: list[float] = []
    if RESUME_CHECKPOINT.exists():
        start_step, history = load_checkpoint(RESUME_CHECKPOINT, model, ema_model, optimizer, scaler)
        print(f"Resuming production training from step {start_step}")
    else:
        # Overfit used a few frozen images. Production starts from a fresh paper-width net.
        model = UNet(cfg).to(device)
        ema_model = copy.deepcopy(model)
        ema_model.requires_grad_(False)
        ema_model.eval()
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        print("Starting production training from random initialization")
    train_seconds = None if TRAIN_MINUTES <= 0 else TRAIN_MINUTES * 60
    print(f"Time cap: {TRAIN_MINUTES} min" if train_seconds else "No time cap; running to max_steps")
    history = train(
        model,
        ema_model,
        optimizer,
        scaler,
        schedule,
        train_loader,
        start_step,
        cfg.max_steps,
        history,
        max_seconds=train_seconds,
    )
    if history:
        print(f"L_simple first/last: {history[0]:.4f} / {history[-1]:.4f}  (n={len(history)})")
else:
    print("Skipping the long run. Set RUN_PRODUCTION_TRAINING = True after overfit succeeds.")


## 8. What this notebook proved

- $L_\text{simple}$ is finite on a real CIFAR-10 batch and starts near 1, as zero-init predicts.
- A frozen $(x_0, t, \varepsilon)$ overfit drives that loss down by more than half.
- Random-$t$ updates on the same images stay finite (the production objective).
- A checkpoint restores model, EMA, optimizer, scaler, step, and loss history; one more step works.
- The production loop is written and resumable. `TRAIN_MINUTES` can stop it on a wall-clock cap.

Next: [`07_sample_eval.ipynb`](07_sample_eval.ipynb) — reverse process, sample grids, curves, one
ablation. Evaluation there is qualitative (no FID in this repo).
